In [ ]:
# This is necessary to recognize the modules
import os
import sys
from decimal import Decimal
import warnings

# warnings.filterwarnings("ignore")

root_path = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
sys.path.append(root_path)

In [ ]:
from core.backtesting import BacktestingEngine

backtesting = BacktestingEngine(root_path=root_path, load_cached_data=True)

In [ ]:
from hummingbot.strategy_v2.executors.position_executor.data_types import TrailingStop
import datetime
from decimal import Decimal
from controllers.directional_trading.pz_scalper import PZScalperControllerConfig


# Controller configuration
connector_name = "binance_perpetual"
trading_pair = "WLD-USDT"
interval = "5m"
backtesting_resolution = "1m"

# Don't matter
cooldown_time = 1 #60 * 15
take_profit = 5 # 100%, -> Disable Take profit, let the trailing do it's job
stop_loss = 5
trailing_stop_activation_price = 1
trailing_stop_trailing_delta = 0.05

# General
total_amount_quote: int = 1000
max_executors_per_side: int = 2


# Indicator Values
hma_fast: int = 10
hma_slow: int = 45
natr_length: int = 21

# Triple Barrier
time_limit: int = 1500
tp_natr_factor = 0.75
sl_natr_factor = 3.0
ts_activation_natr_factor = 1.0
ts_delta_natr_factor = 0.75
###


# Creating the instance of the configuration and the controller
config = PZScalperControllerConfig(
    connector_name=connector_name,
    trading_pair=trading_pair,
    interval=interval,
    take_profit=Decimal(take_profit),
    stop_loss=Decimal(stop_loss),
    trailing_stop=TrailingStop(activation_price=Decimal(trailing_stop_activation_price), trailing_delta=Decimal(trailing_stop_trailing_delta)),
    total_amount_quote=Decimal(total_amount_quote),
    time_limit=time_limit,
    max_executors_per_side=max_executors_per_side,
    cooldown_time=cooldown_time,
    natr_length = natr_length,
    sl_natr_factor=sl_natr_factor,
    ts_activation_natr_factor = ts_activation_natr_factor,
    ts_delta_natr_factor = ts_delta_natr_factor,
    tp_natr_factor=tp_natr_factor,
    hma_fast=hma_fast,
    hma_slow=hma_slow,
)

In [ ]:
# Running the backtesting this will output a backtesting result object that has built in methods to visualize the results

start = int(datetime.datetime(2025, 2, 5).timestamp())
end = int(datetime.datetime(2025, 3, 30).timestamp())

backtesting_result = await backtesting.run_backtesting(config, start, end, backtesting_resolution)

In [ ]:
import plotly.graph_objects as go
# from plotly.subplots import make_subplots

# Let's see what is inside the backtesting results
print(backtesting_result.get_results_summary())
fig = backtesting_result.get_backtesting_figure()
# Add EMAs
candles_df = backtesting_result.processed_data

fast_key = f"HMA_{hma_fast}"
slow_key = f"HMA_{hma_slow}"


fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[fast_key],
                         line=dict(color='#00FF00', width=2),
                         name='Fast HMA'))
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[slow_key],
                         line=dict(color='#FF0000', width=2),
                         name='Fast HMA'))


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Grab processed data
candles_df = backtesting_result.processed_data

# Create subplots layout
fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    row_heights=[0.60, 0.15, 0.15, 0.1],
    vertical_spacing=0.02,
    subplot_titles=["Price + Indicators", "Condition", "Crossover"]
)

# Row 1: Close price
fig.add_trace(go.Scatter(
    x=candles_df.index, y=candles_df["close"],
    line=dict(color='#FFFFFF', width=2), name='Close'), row=1, col=1)


hma_fast_key = f"HMA_{hma_fast}"
hma_slow_key = f"HMA_{hma_slow}"
# ema_medium_key = f"EMA_{ema_medium}"
# ema_slow_key = f"EMA_{ema_slow}"


fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[hma_fast_key],
                         line=dict(color='#00FF00', width=2),
                         name='Fast HMA'))
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[hma_slow_key],
                         line=dict(color='#FF0000', width=2),
                         name='Slow HMA'))
# fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[ema_medium_key],
#                          line=dict(color='#FFA500', width=2),
#                          name='Slow HMA'))
# fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[ema_slow_key],
#                          line=dict(color='#FFFFFF', width=2),
#                          name='Slow EMA'))

# Row 4: Signal values as line + markers
# fig.add_trace(go.Scatter(
#     x=candles_df.index,
#     y=candles_df[f"STOCHRSId_{srsi_length}_{srsi_length}_{srsi_smoothing}_{srsi_smoothing}"],
#     mode='lines',
#     line=dict(color='red', width=2),
#     name='short_condition',
#     marker=dict(size=6)
# ), row=2, col=1)
# fig.add_trace(go.Scatter(
#     x=candles_df.index,
#     y=candles_df[f"STOCHRSIk_{srsi_length}_{srsi_length}_{srsi_smoothing}_{srsi_smoothing}"],
#     mode='lines',
#     line=dict(color='green', width=2),
#     name='short_condition',
#     marker=dict(size=6)
# ), row=2, col=1)

# fig.add_trace(go.Scatter(
#     x=candles_df.index,
#     y=candles_df["bearish_crossover"],
#     mode='lines+markers',
#     line=dict(color='cyan', width=2),
#     name='bearish_crossover',
#     marker=dict(size=6)
# ), row=3, col=1)
fig.add_trace(go.Scatter(
    x=candles_df.index,
    y=candles_df["signal"],
    mode='lines+markers',
    line=dict(color='cyan', width=2),
    name='signal',
    marker=dict(size=6)
), row=3, col=1)


# Final layout
fig.update_layout(
    height=700,
    title="Backtest with Candles, HMAs, RSI, and StochRSI",
    showlegend=True,
    template="plotly_dark"
)

fig.show()



In [ ]:
# # 2. The executors dataframe: this is the dataframe that contains the information of the orders that were executed
import pandas as pd

executors_df = backtesting_result.executors_df
executors_df

### Backtesting Analysis

### Scatter of PNL per Trade
This bar chart illustrates the PNL for each individual trade. Positive PNLs are shown in green and negative PNLs in red, providing a clear view of profitable vs. unprofitable trades.


In [ ]:
# import plotly.express as px

# # Create a new column for profitability
# executors_df['profitable'] = executors_df['net_pnl_quote'] > 0

# # Create the scatter plot
# fig = px.scatter(
#     executors_df,
#     x="timestamp",
#     y='net_pnl_quote',
#     title='PNL per Trade',
#     color='profitable',
#     color_discrete_map={True: 'green', False: 'red'},
#     labels={'timestamp': 'Timestamp', 'net_pnl_quote': 'Net PNL (Quote)'},
#     hover_data=['filled_amount_quote', 'side']
# )

# # Customize the layout
# fig.update_layout(
#     xaxis_title="Timestamp",
#     yaxis_title="Net PNL (Quote)",
#     legend_title="Profitable",
#     font=dict(size=12, color="white"),
#     showlegend=False,
#     plot_bgcolor='rgba(0,0,0,0.8)',  # Dark background
#     paper_bgcolor='rgba(0,0,0,0.8)',  # Dark background for the entire plot area
#     xaxis=dict(gridcolor="gray"),
#     yaxis=dict(gridcolor="gray")
# )

# # Add a horizontal line at y=0 to clearly separate profits and losses
# fig.add_hline(y=0, line_dash="dash", line_color="lightgray")

# # Show the plot
# fig.show()

### Histogram of PNL Distribution
The histogram displays the distribution of PNL values across all trades. It helps in understanding the frequency and range of profit and loss outcomes.


In [ ]:
# fig = px.histogram(executors_df, x='net_pnl_quote', title='PNL Distribution')
# fig.show()


# Conclusion
We can see that the indicator has potential to bring good signals to trade and might be interesting to see how we can design a market maker that shifts the mid price based on this indicator.
A lot of the short signals are wrong but if we zoom in into the loss signals we can see that the losses are not that big and the wins are bigger and if we had implemented the trailing stop feature probably a lot of them are going to be profits.

# Next steps
- Filter only the loss signals and understand what you can do to prevent them
- Try different configuration values for the indicator
- Test in multiple markets, pick mature markets like BTC-USDT or ETH-USDT and also volatile markets like DOGE-USDT or SHIB-USDT